## Piloto local Qwen

**Guía de ejecución:** Requiere GPU compatible y datos de data/qwen_micro_pilot. Es un experimento independiente: no confundir sus fuentes con el piloto definitivo de Visual Studio.

Instala las dependencias una vez desde la raíz del repositorio con `pip install -e ".[notebooks]"`. Los archivos de `data/` se mantienen fuera de Git. Este notebook se conserva para reproducir la fase de investigación; no se recomienda pulsar «Ejecutar todo» sin revisar sus fases y requisitos.


**Versión adaptada para el repositorio unificado.** Los módulos reutilizables se importan desde `src/bias_detector/`; los datos pesados se guardan bajo `data/` y no se descargan automáticamente. Ejecutar el notebook en orden.

In [ ]:
# Preparación: iniciar Jupyter desde la raíz del repositorio (local o Colab).
from pathlib import Path
import os, sys

def _locate_repo():
    for folder in (Path.cwd(), *Path.cwd().parents, Path('/content/cognitive-bias-detector')):
        if (folder / 'pyproject.toml').is_file() and (folder / 'src' / 'bias_detector').is_dir():
            return folder.resolve()
    raise RuntimeError('No se encontró la raíz del repositorio. Abre Jupyter allí o usa %cd /content/cognitive-bias-detector en Colab.')

PROJECT_ROOT = _locate_repo()
source_path = str(PROJECT_ROOT / 'src')
if source_path not in sys.path:
    sys.path.insert(0, source_path)
from bias_detector.config import (
    DATA_DIR, MODEL_DATA_DIR, MODEL_DIR, PROMPTS_DIR, RESULTS_DIR,
    RAW_DIR, PROCESSED_DIR, MODELING_DIR, PILOT_DIR,
    REDACTION_ANALYSIS_DIR, ANNOTATION_RESULTS_DIR,
    MASS_ANNOTATION_DIR, FINAL_ANNOTATION_DIR, PILOT_RESULTS_DIR,
    MICRO_PILOT_RESULTS_DIR, CORPUS_DATA, FINAL_ANNOTATIONS_DATA,
    require_file, ensure_output_dir,
)
print('Repositorio:', PROJECT_ROOT)
print('Directorio de datos:', DATA_DIR)


# Anotación asistida por LLM

En este notebook, se va a realizar la anotación de las pruebas piloto y del corpus final de conversaciones con el modelo abierto Qwen3-8B.

## 1. Anotación del conjunto micro-piloto (10 conversaciones, 40 llamadas)

### Configuración inicial

In [ ]:
# Comprobamos que la GPU está activada para este entorno
!nvidia-smi

In [ ]:
# Instalación de dependencias
%pip install -q -U \
    "transformers>=4.51.0" \
    accelerate \
    bitsandbytes \
    pandas \
    pydantic

In [ ]:
# Debe ejecutarse antes de importar torch
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True"
)

In [ ]:
# Imports
from pathlib import Path
from enum import Enum
from datetime import datetime, timezone

import json
import time
import re
import gc

import pandas as pd
import torch

from pydantic import BaseModel, Field, ValidationError
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

In [ ]:
# Comprobación de entorno
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    props = torch.cuda.get_device_properties(0)

    print(
        "VRAM:",
        round(
            props.total_memory / 1024**3,
            2
        ),
        "GB"
    )

In [ ]:
# Archivos Qwen: datos locales del repositorio (sin Google Drive)
BASE_DIR = DATA_DIR / 'qwen_micro_pilot'
DATA_PATH = BASE_DIR / 'micro_pilot_dataset.pkl'
PROMPT_DIR = PROMPTS_DIR
RESULTS_DIR = PROJECT_ROOT / 'results' / 'qwen_micro_pilot'
RESULTS_DIR.mkdir(parents=True,exist_ok=True)

### Carga del micro-pilot

In [ ]:
micro_pilot_df = pd.read_pickle(DATA_PATH)

print(f"Conversaciones cargadas: "f"{len(micro_pilot_df)}")

assert len(micro_pilot_df) == 10
assert micro_pilot_df["conversation_id"].is_unique

micro_pilot_df[
    [
        "conversation_id",
        "n_messages",
        "length_bin"
    ]
]

### Carga de los prompts

In [ ]:
# Rutas y versionado
PROMPT_VERSION = "v0.2"

PROMPT_FILES = {
    "confirmation_bias": PROMPT_DIR / "confirmation_bias.txt",
    "anchoring_bias": PROMPT_DIR / "anchoring_bias.txt",
    "framing_effect": PROMPT_DIR / "framing_effect.txt",
    "availability_bias": PROMPT_DIR / "availability_bias.txt"
}

In [ ]:
from bias_detector.annotation_utils import load_prompt_templates
PROMPTS = load_prompt_templates(PROMPT_FILES)


In [ ]:
# Comprobación de que tenemos los mismos prompts
for bias, prompt in PROMPTS.items():
    print(
        f"{bias}: "
        f"{len(prompt):,} caracteres"
    )

In [ ]:
from bias_detector.annotation_utils import format_annotation_prompt
def build_prompt(bias, conversation_text):
    return format_annotation_prompt(PROMPTS, bias, conversation_text)


### Esquema común de salida

In [ ]:
class AnnotationLabel(str, Enum):
    PRESENT = "PRESENT"
    ABSENT = "ABSENT"
    UNCERTAIN = "UNCERTAIN"


class ConfidenceLevel(str, Enum):
    HIGH = "HIGH"
    MEDIUM = "MEDIUM"
    LOW = "LOW"


class BiasAnnotation(BaseModel):
    bias: str
    label: AnnotationLabel
    trigger_turns: list[int] = Field(default_factory=list)
    evidence_turns: list[int] = Field(default_factory=list)
    justification: str
    confidence: ConfidenceLevel

Aquí hay una diferencia importante respecto a OpenAI: Qwen no nos va a proporcionar Structured Outputs mediante Pydantic automáticamente. Por tanto:
1. generamos texto;
2. extraemos el JSON;
3. lo validamos con Pydantic.

Eso es precisamente una de las cosas interesantes que queremos medir: su tasa de cumplimiento del formato.

In [ ]:
from bias_detector.annotation_utils import validate_annotation_turns
def validate_annotation(annotation, expected_bias, n_messages):
    return validate_annotation_turns(annotation, expected_bias, n_messages)

def validate_annotation_logic(
    annotation
):

    if (
        annotation.label
        == AnnotationLabel.PRESENT
        and not annotation.evidence_turns
    ):
        raise ValueError(
            "PRESENT debe incluir "
            "evidence_turns."
        )

    return True

### Configuración de Qwen: cuantización a 4 bits

La cuantización consiste en representar los pesos del modelo con menos precisión numérica. En vez de almacenar cada peso con 16 bits, por ejemplo, se utilizan 4 bits. Eso reduce mucho la memoria necesaria para cargar el modelo y, normalmente, también facilita la inferencia en GPUs con VRAM limitada.

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [ ]:
# Carga de Qwen3-8B
MODEL_NAME = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# Modelo
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype="auto"
)

model.eval()

In [ ]:
print("Modelo cargado en:", model.device)

In [ ]:
!nvidia-smi

### Configuración de LLM_B

In [ ]:
ANNOTATOR_CONFIG = {
    "annotator_name": "LLM_B",
    "provider": "local_huggingface",
    "model": MODEL_NAME,
    "quantization": "4bit_nf4",
    "thinking": False # Obtenemos mayor eficiencia y tenemos unos manuales de inferencia estructurados
}

### Función básica de inferencia

In [ ]:
def call_qwen_annotator(
    prompt,
    max_new_tokens=256
):

    # --------------------------------------------------
    # Limpieza previa de memoria
    # --------------------------------------------------
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --------------------------------------------------
    # Preparar input
    # --------------------------------------------------
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    model_inputs = tokenizer(
        formatted_text,
        return_tensors="pt"
    ).to(model.device)

    input_tokens = int(
        model_inputs["input_ids"].shape[-1]
    )

    # --------------------------------------------------
    # Función auxiliar de generación
    # --------------------------------------------------
    def run_generation(use_cache):

        with torch.inference_mode():

            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=use_cache,
                pad_token_id=tokenizer.eos_token_id
            )

        return generated_ids

    # --------------------------------------------------
    # Primer intento: configuración normal
    # --------------------------------------------------
    try:

        generated_ids = run_generation(
            use_cache=True
        )

        generation_mode = (
            "normal_cache"
        )

    # --------------------------------------------------
    # Fallback si ocurre CUDA OOM
    # --------------------------------------------------
    except torch.OutOfMemoryError:

        print(
            "    CUDA OOM detectado. "
            "Reintentando con use_cache=False..."
        )

        # Eliminar posibles referencias/tensores
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        try:

            generated_ids = run_generation(
                use_cache=False
            )

            generation_mode = (
                "fallback_no_cache"
            )

        except torch.OutOfMemoryError:

            # Limpiar otra vez antes de propagar
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            raise

    # --------------------------------------------------
    # Extraer únicamente tokens generados
    # --------------------------------------------------
    output_ids = generated_ids[
        0,
        model_inputs["input_ids"].shape[-1]:
    ]

    output_tokens = int(
        output_ids.shape[-1]
    )

    output_text = tokenizer.decode(
        output_ids,
        skip_special_tokens=True
    ).strip()

    total_tokens = (
        input_tokens
        + output_tokens
    )

    # --------------------------------------------------
    # Limpieza posterior
    # --------------------------------------------------
    del generated_ids
    del output_ids
    del model_inputs

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "text": output_text,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "generation_mode": generation_mode
    }

In [ ]:
# Prueba mínima
test_row = micro_pilot_df.iloc[0]

test_prompt = build_prompt(
    bias="confirmation_bias",
    conversation_text=(
        test_row["conversation_text"]
    )
)

# Llamada
test_generation = (
    call_qwen_annotator(
        test_prompt
    )
)

print(
    test_generation["text"]
)

### Constructor del parser

In [ ]:
def clean_model_output(text):
    """
    Limpia envoltorios frecuentes como bloques Markdown ```json ... ```
    y espacios sobrantes.
    """
    if text is None:
        return ""

    text = text.strip()

    # Eliminar fences Markdown si existen
    if text.startswith("```"):
        text = re.sub(
            r"^```(?:json)?\s*",
            "",
            text,
            flags=re.IGNORECASE
        )

        text = re.sub(
            r"\s*```$",
            "",
            text
        )

    return text.strip()

In [ ]:
def extract_json_object(text):
    """
    Intenta recuperar el primer objeto JSON completo contenido
    en la respuesta del modelo.
    """

    text = clean_model_output(text)

    # Primer intento: toda la respuesta es JSON válido
    try:
        return json.loads(text)

    except json.JSONDecodeError:
        pass

    # Segundo intento: localizar el primer '{'
    start = text.find("{")

    if start == -1:
        raise ValueError(
            "No se encontró ningún objeto JSON en la respuesta."
        )

    # Recorremos el texto contando llaves para localizar
    # el cierre del primer objeto JSON completo.
    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):

        char = text[i]

        if escape:
            escape = False
            continue

        if char == "\\":
            escape = True
            continue

        if char == '"':
            in_string = not in_string
            continue

        if not in_string:

            if char == "{":
                depth += 1

            elif char == "}":
                depth -= 1

                if depth == 0:

                    candidate = text[start:i + 1]

                    try:
                        return json.loads(candidate)

                    except json.JSONDecodeError as e:
                        raise ValueError(
                            "Se encontró un bloque con forma de JSON, "
                            "pero no pudo parsearse."
                        ) from e

    raise ValueError(
        "Se encontró el inicio de un JSON, pero no un cierre válido."
    )

In [ ]:
# Validación contra el esquema 'BiasAnnotation'
def parse_qwen_annotation(
    raw_text,
    expected_bias,
    n_messages
):
    """
    Extrae, valida estructuralmente y valida semánticamente
    una anotación generada por Qwen.
    """

    data = extract_json_object(
        raw_text
    )

    try:
        annotation = BiasAnnotation.model_validate(
            data
        )

    except ValidationError as e:
        raise ValueError(
            "El JSON es válido, pero no cumple "
            "el esquema BiasAnnotation."
        ) from e

    validate_annotation(
        annotation=annotation,
        expected_bias=expected_bias,
        n_messages=n_messages
    )

    validate_annotation_logic(
        annotation=annotation
    )

    return annotation

In [ ]:
# Probamos con la salida que acabamos de obtener con Qwen
raw_qwen_output = test_generation["text"]

parsed_annotation = parse_qwen_annotation(
    raw_text=raw_qwen_output,
    expected_bias="confirmation_bias",
    n_messages=int(test_row["n_messages"])
)

parsed_annotation

In [ ]:
print(parsed_annotation.label.value)
print(parsed_annotation.confidence.value)
print(parsed_annotation.trigger_turns)
print(parsed_annotation.evidence_turns)
print(parsed_annotation.justification)

### Función de llamada de inferencia

In [ ]:
# A diferencia del otro anotador, en este caso guardamos la respuesta
# original ('raw_response') para ver si alguna inferencia
# falla por formato
def annotate_single_bias_qwen(
    row,
    bias,
    annotator_config
):
    conversation_id = row["conversation_id"]
    n_messages = int(row["n_messages"])

    prompt = build_prompt(
        bias=bias,
        conversation_text=row["conversation_text"]
    )

    start_time = time.perf_counter()

    try:

        generation = call_qwen_annotator(
            prompt=prompt
        )

        latency = (
            time.perf_counter()
            - start_time
        )

        raw_text = generation["text"]

        try:

            data = extract_json_object(
                raw_text
            )

        except Exception as e:

            return {
                "conversation_id": conversation_id,
                "annotator": annotator_config["annotator_name"],
                "provider": annotator_config["provider"],
                "model": annotator_config["model"],
                "bias": bias,
                "label": None,
                "confidence": None,
                "trigger_turns": None,
                "evidence_turns": None,
                "justification": None,
                "status": "ERROR",
                "error_type": "INVALID_JSON",
                "error_message": str(e),
                "latency_seconds": latency,
                "input_tokens": generation["input_tokens"],
                "output_tokens": generation["output_tokens"],
                "total_tokens": generation["total_tokens"],
                "generation_mode": generation["generation_mode"],
                "raw_response": raw_text
            }

        try:

            annotation = (
                BiasAnnotation.model_validate(
                    data
                )
            )

        except ValidationError as e:

            return {
                "conversation_id": conversation_id,
                "annotator": annotator_config["annotator_name"],
                "provider": annotator_config["provider"],
                "model": annotator_config["model"],
                "bias": bias,
                "label": None,
                "confidence": None,
                "trigger_turns": None,
                "evidence_turns": None,
                "justification": None,
                "status": "ERROR",
                "error_type": "INVALID_SCHEMA",
                "error_message": str(e),
                "latency_seconds": latency,
                "input_tokens": generation["input_tokens"],
                "output_tokens": generation["output_tokens"],
                "total_tokens": generation["total_tokens"],
                "generation_mode": generation["generation_mode"],
                "raw_response": raw_text
            }

        # Validaciones semánticas
        validate_annotation(
            annotation=annotation,
            expected_bias=bias,
            n_messages=n_messages
        )

        validate_annotation_logic(
            annotation=annotation
        )

        return {
            "conversation_id": conversation_id,
            "annotator": annotator_config["annotator_name"],
            "provider": annotator_config["provider"],
            "model": annotator_config["model"],
            "bias": bias,
            "label": annotation.label.value,
            "confidence": annotation.confidence.value,
            "trigger_turns": annotation.trigger_turns,
            "evidence_turns": annotation.evidence_turns,
            "justification": annotation.justification,
            "status": "SUCCESS",
            "error_type": None,
            "error_message": None,
            "latency_seconds": latency,
            "input_tokens": generation["input_tokens"],
            "output_tokens": generation["output_tokens"],
            "total_tokens": generation["total_tokens"],
            "generation_mode": generation["generation_mode"],
            "raw_response": raw_text
        }

    except Exception as e:

        latency = (
            time.perf_counter()
            - start_time
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return {
            "conversation_id": conversation_id,
            "annotator": annotator_config["annotator_name"],
            "provider": annotator_config["provider"],
            "model": annotator_config["model"],
            "bias": bias,
            "label": None,
            "confidence": None,
            "trigger_turns": None,
            "evidence_turns": None,
            "justification": None,
            "status": "ERROR",
            "error_type": type(e).__name__,
            "error_message": str(e),
            "latency_seconds": latency,
            "input_tokens": None,
            "output_tokens": None,
            "total_tokens": None,
            "generation_mode": None,
            "raw_response": None
        }

### Segunda prueba mínima

In [ ]:
test_qwen_result = annotate_single_bias_qwen(
    row=test_row,
    bias="confirmation_bias",
    annotator_config=ANNOTATOR_CONFIG
)

test_qwen_result

In [ ]:
# Segunda prueba mínima con otra conversación
test_row_2 = micro_pilot_df.iloc[1]

test_qwen_result_2 = annotate_single_bias_qwen(
    row=test_row_2,
    bias="framing_effect",
    annotator_config=ANNOTATOR_CONFIG
)

test_qwen_result_2

### Ejecución de las 40 llamadas

In [ ]:
# Configuración inicial
PROMPT_VERSION = "v0.2"

BIASES = [
    "confirmation_bias",
    "anchoring_bias",
    "framing_effect",
    "availability_bias"
]

OUTPUT_FILE_QWEN = (
    RESULTS_DIR
    / f"micro_pilot_llm_b_{PROMPT_VERSION.replace('.', '_')}.jsonl"
)

print(f"Archivo de resultados: {OUTPUT_FILE_QWEN}")

In [ ]:
# Función para el guardado incremental
def append_result_jsonl(
    result,
    output_file
):
    with open(
        output_file,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            json.dumps(
                result,
                ensure_ascii=False
            )
            + "\n"
        )

In [ ]:
# Recuperación de ejecuciones anteriores
def load_previous_results(
    output_file
):
    if not output_file.exists():
        return pd.DataFrame()

    try:
        return pd.read_json(
            output_file,
            lines=True
        )

    except ValueError:
        return pd.DataFrame()

# Recuperación de llamadas exitosas
def get_successful_pairs(
    results_df
):
    if results_df.empty:
        return set()

    successful = results_df[
        results_df["status"] == "SUCCESS"
    ]

    return set(
        zip(
            successful["conversation_id"],
            successful["bias"]
        )
    )

# Conteo de intentos
def get_attempt_counts(
    results_df
):
    if results_df.empty:
        return {}

    return (
        results_df
        .groupby(
            [
                "conversation_id",
                "bias"
            ]
        )
        .size()
        .to_dict()
    )

In [ ]:
# Bucle de llamada
previous_results_df = load_previous_results(OUTPUT_FILE_QWEN)

successful_pairs = get_successful_pairs(previous_results_df)

attempt_counts = get_attempt_counts(previous_results_df)

total_tasks = len(micro_pilot_df) * len(BIASES)

current_task = 0

for _, row in micro_pilot_df.iterrows():

    conversation_id = row["conversation_id"]

    for bias in BIASES:

        current_task += 1

        key = (conversation_id, bias)

        if key in successful_pairs:

            print(
                f"[{current_task:02d}/{total_tasks}] "
                f"{conversation_id} | {bias} "
                f"-> SKIPPED"
            )

            continue

        attempt_number = attempt_counts.get(key,0) + 1

        print(
            f"[{current_task:02d}/{total_tasks}] "
            f"{conversation_id} | {bias} "
            f"| attempt {attempt_number}"
        )

        result = annotate_single_bias_qwen(
            row=row,
            bias=bias,
            annotator_config=ANNOTATOR_CONFIG
        )

        result["prompt_version"] = PROMPT_VERSION

        result["attempt"] = attempt_number

        result["timestamp_utc"] = datetime.now(timezone.utc).isoformat()

        append_result_jsonl(result=result,output_file=OUTPUT_FILE_QWEN)

        attempt_counts[key] = attempt_number

        if result["status"] == "SUCCESS":

            successful_pairs.add(key)

            print(
                f"    SUCCESS | "
                f"{result['label']} | "
                f"{result['confidence']} | "
                f"{result['latency_seconds']:.2f}s | "
                f"{result['total_tokens']} tokens"
            )

        else:

            print(
                f"    ERROR | "
                f"{result['error_type']} | "
                f"{result['error_message']}"
            )

        # No es imprescindible al ser inferencia local,
        # pero deja respirar un poco a la sesión.
        time.sleep(0.2)

In [ ]:
# Cargamos el historial
history_qwen_df = pd.read_json(
    OUTPUT_FILE_QWEN,
    lines=True
)

print(
    f"Intentos registrados: "
    f"{len(history_qwen_df)}"
)

In [ ]:
# Construimos la versión definitiva de una fila por combinación
success_qwen_df = (
    history_qwen_df[
        history_qwen_df["status"]
        == "SUCCESS"
    ]
    .sort_values("attempt")
    .drop_duplicates(
        subset=[
            "conversation_id",
            "bias"
        ],
        keep="last"
    )
)

In [ ]:
assert len(success_qwen_df) == 40

In [ ]:
# Guardams CSV y PKL
FINAL_CSV_QWEN = (
    RESULTS_DIR
    / "micro_pilot_llm_b_v0_2.csv"
)

FINAL_PKL_QWEN = (
    RESULTS_DIR
    / "micro_pilot_llm_b_v0_2.pkl"
)

success_qwen_df.to_csv(
    FINAL_CSV_QWEN,
    index=False,
    encoding="utf-8"
)

success_qwen_df.to_pickle(
    FINAL_PKL_QWEN
)

In [ ]:
# Cargar el dataset recién guardado como punto de control
success_qwen_df = pd.read_pickle(
    FINAL_PKL_QWEN
)

qwen_results_df = pd.read_pickle(
    RESULTS_DIR / "micro_pilot_llm_b_v0_2.pkl"
)

### Análisis de resultados

Mismo diagnóstico que con el anotador Luna.

In [ ]:
print(
    pd.crosstab(
        success_qwen_df["bias"],
        success_qwen_df["label"],
        margins=True
    )
)

In [ ]:
print(
    pd.crosstab(
        success_qwen_df["bias"],
        success_qwen_df["confidence"],
        margins=True
    )
)

In [ ]:
print(
    success_qwen_df[
        "latency_seconds"
    ].describe()
)

In [ ]:
print(
    "Input tokens:",
    success_qwen_df[
        "input_tokens"
    ].sum()
)

print(
    "Output tokens:",
    success_qwen_df[
        "output_tokens"
    ].sum()
)

print(
    "Total tokens:",
    success_qwen_df[
        "total_tokens"
    ].sum()
)

## Anotación del conjunto piloto (200 conversaciones)

### Configuración inicial

In [ ]:
# Muestra piloto: colocar la versión original aquí si se necesita repetir el experimento
BASE_DIR = DATA_DIR / 'qwen_micro_pilot'
PILOT_PATH = BASE_DIR / 'pilot_dataset.pkl'
pilot_df = pd.read_pickle(require_file(PILOT_PATH,'muestra piloto Qwen'))
print(f'Conversaciones cargadas: {len(pilot_df)}')
assert len(pilot_df)==200
assert pilot_df['conversation_id'].is_unique
assert pilot_df['conversation_text'].notna().all()
pilot_df[['conversation_id','n_messages','length_bin']].head()

In [ ]:
# Configuración del anotador
PROMPT_VERSION = "v0.2"

BIASES = [
    "confirmation_bias",
    "anchoring_bias",
    "framing_effect",
    "availability_bias"
]

ANNOTATOR_CONFIG = {
    "annotator_name": "LLM_B",
    "provider": "local_huggingface",
    "model": "Qwen/Qwen3-8B",
    "quantization": "4bit_nf4",
    "thinking": False
}

TOTAL_TASKS = len(pilot_df) * len(BIASES)

print(f"Total de anotaciones: "f"{TOTAL_TASKS}")

assert TOTAL_TASKS == 800

In [ ]:
# Salida local persistente dentro del proyecto (sin Google Drive)
RESULTS_DIR = PROJECT_ROOT / 'results' / 'qwen_micro_pilot' / 'pilot'
RESULTS_DIR.mkdir(parents=True,exist_ok=True)
OUTPUT_FILE_QWEN = RESULTS_DIR / 'pilot_llm_b_v0_2.jsonl'
print(OUTPUT_FILE_QWEN)

In [ ]:
# Rcuperamos progreso previo
previous_results_df = (
    load_previous_results(
        OUTPUT_FILE_QWEN
    )
)

successful_pairs = (
    get_successful_pairs(
        previous_results_df
    )
)

attempt_counts = (
    get_attempt_counts(
        previous_results_df
    )
)

print(
    f"Anotaciones ya completadas: "
    f"{len(successful_pairs)} / "
    f"{TOTAL_TASKS}"
)

Si Colab se desconectase en medio de la sesión, podemos recuperar el trabajo de las anotaciones por donde se haya quedado.

### Bucle de inferencia

In [ ]:
# Bucle de llamada
current_task = 0

for _, row in pilot_df.iterrows():

    conversation_id = row["conversation_id"]

    for bias in BIASES:

        current_task += 1

        key = (conversation_id, bias)

        # Saltar la conversación si ya se completó
        if key in successful_pairs:

            print(
                f"[{current_task:02d}/{TOTAL_TASKS}] "
                f"{conversation_id} | {bias} "
                f"-> SKIPPED"
            )

            continue

        # Número de intento
        attempt_number = attempt_counts.get(key,0) + 1

        print(
            f"[{current_task:02d}/{TOTAL_TASKS}] "
            f"{conversation_id} | {bias} "
            f"| attempt {attempt_number}"
        )

        # Inferencia
        result = annotate_single_bias_qwen(
            row=row,
            bias=bias,
            annotator_config=ANNOTATOR_CONFIG
        )

        # Metadatos
        result["prompt_version"] = PROMPT_VERSION

        result["attempt"] = attempt_number

        result["timestamp_utc"] = datetime.now(timezone.utc).isoformat()

        # Guardado incremental
        append_result_jsonl(result=result,output_file=OUTPUT_FILE_QWEN)

        attempt_counts[key] = attempt_number

        # Feedback
        if result["status"] == "SUCCESS":

            successful_pairs.add(key)

            print(
                f"    SUCCESS | "
                f"{result['label']} | "
                f"{result['confidence']} | "
                f"{result['latency_seconds']:.2f}s | "
                f"{result['total_tokens']} tokens"
            )

        else:

            print(
                f"    ERROR | "
                f"{result['error_type']} | "
                f"{result['error_message']}"
            )

        # No es imprescindible al ser inferencia local,
        # pero deja respirar un poco a la sesión.
        time.sleep(0.2)

In [ ]:
# Construimos el resultado final
history_qwen_df = pd.read_json(
    OUTPUT_FILE_QWEN,
    lines=True
)

print(
    f"Intentos registrados: "
    f"{len(history_qwen_df)}"
)

In [ ]:
success_qwen_df = (
    history_qwen_df[
        history_qwen_df["status"]
        == "SUCCESS"
    ]
    .sort_values("attempt")
    .drop_duplicates(
        subset=[
            "conversation_id",
            "bias"
        ],
        keep="last"
    )
)

In [ ]:
# Para este caso, no se han podido completar todas las anotatciones por errores de memoria
print(
    f"Anotaciones correctas: "
    f"{len(success_qwen_df)} / 800"
)

In [ ]:
annotations_per_conversation = success_qwen_df.groupby("conversation_id").size()

print(annotations_per_conversation.value_counts())

Tenemos 188 conversaciones con 4 anotaciones y 1 conversación con 2 anotaciones completadas.

In [ ]:
print(success_qwen_df["bias"].value_counts())

In [ ]:
# Guardamos resultados
FINAL_CSV_QWEN = (
    RESULTS_DIR
    / "pilot_llm_b_v0_2.csv"
)

FINAL_PKL_QWEN = (
    RESULTS_DIR
    / "pilot_llm_b_v0_2.pkl"
)

success_qwen_df.to_csv(
    FINAL_CSV_QWEN,
    index=False,
    encoding="utf-8"
)

success_qwen_df.to_pickle(
    FINAL_PKL_QWEN
)

print("Resultados finales guardados.")

### Análisis de resultados

In [ ]:
# Distribución de etiquetas
print(
    pd.crosstab(
        success_qwen_df["bias"],
        success_qwen_df["label"],
        margins=True
    )
)

In [ ]:
# Confianza
print(
    pd.crosstab(
        success_qwen_df["bias"],
        success_qwen_df["confidence"],
        margins=True
    )
)

In [ ]:
# Prevalencia provisional
qwen_label_distribution = (
    success_qwen_df
    .groupby("bias")["label"]
    .value_counts(
        normalize=True
    )
    .mul(100)
    .round(2)
)

print(qwen_label_distribution)